In [77]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
# models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [78]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [79]:
def extract_numerical_text(df):
  num_cols = df.select_dtypes(include=['int16', 'int32', 'int64', 'float32', 'float64']).columns
  text_cols = df.select_dtypes(exclude=['int16', 'int32', 'int64', 'float32', 'float64']).columns

  df_num = df[num_cols]
  df_text = df[text_cols]

  return df_num, df_text, num_cols, text_cols

In [80]:
def combineDFs(df_1, df_2):
  df = pd.concat([df_1, df_2], axis=1)

  return df

In [81]:
def move_label_column(df, label_col_number):
  if label_col_number != -1:
    cols = list(range(df.shape[1]))
    col = cols.pop(label_col_number)
    cols.append(col)
    df = df.iloc[:, cols]

  return df

In [82]:
def section(title):
    print("\n" + "=" * 50)
    print(title)
    print("=" * 50)

def subsection(title):
    print("\n" + "-" * 50)
    print(title)
    print("-" * 50)

def success(msg):
    print(f"✔ {msg}")

In [83]:
# Handling Missing Values
def handle_missing_values(df, method):
  if method == "drop":
    df.dropna(inplace=True)
    success("Missing values dropped")

  elif method == "impute":
    df_num, df_text, num_cols, text_cols = extract_numerical_text(df)

    imputer = KNNImputer(n_neighbors=3)
    df_num_imputed = imputer.fit_transform(df_num)
    df_num = pd.DataFrame(df_num_imputed, columns=num_cols)

    df = combineDFs(df_num, df_text)
    success("Impute")

  return df

In [84]:
# Data Standardizatoin
def Standardize(X_train, X_test):
  num_cols = X_train.select_dtypes(include=['int16', 'int32', 'int64', 'float32', 'float64']).columns
  scaler = StandardScaler()
  X_train = scaler.fit_transform(X_train)
  X_test = scaler.transform(X_test)

  X_train = pd.DataFrame(X_train, columns=num_cols)
  X_test = pd.DataFrame(X_test, columns=num_cols)
  success("Standardization applied")

  return X_train, X_test

In [85]:
# Label encoding
def encodeLabels(Y):
  le = LabelEncoder()
  Y = le.fit_transform(Y)
  success("Labels encoded")

  return le, Y

In [86]:
# Train test split
def split(df, label_col_number, testSize, randomState):
  df = move_label_column(df, label_col_number)

  X = df.iloc[:, :-1]
  Y = df.iloc[:, -1]

  le, Y = encodeLabels(Y)

  X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=testSize, stratify=Y, random_state=randomState)
  success("Train-test split completed")

  return df, X, Y, X_train, Y_train, X_test, Y_test, le

In [87]:
# Handle imbalance Dataset
def handle_imbalace(X_train, Y_train, handleImbalance):
  if handleImbalance == 1 or handleImbalance == "under":
    under_sampler = RandomUnderSampler(sampling_strategy='auto', random_state=69)
    X_train, Y_train = under_sampler.fit_resample(X_train, Y_train)
    success("Undersampling applied")

  elif handleImbalance == 2 or handleImbalance == "over":
    over_sampler = RandomOverSampler(sampling_strategy='auto', random_state=69)
    X_train, Y_train = over_sampler.fit_resample(X_train, Y_train)
    success("Oversampling applied")

  elif handleImbalance == 3 or handleImbalance == "smote":
    decision = input("WARNING! this removes text columns. do You need still Continue(Y / N): ")

    if decision.lower() == 'y':
      X_train_num, _, _, _ = extract_numerical_text(pd.DataFrame(X_train))

      smote = SMOTE(random_state=69, k_neighbors=1)
      X_train, Y_train = smote.fit_resample(X_train_num, Y_train)
      success("Smote applied")

    else :
      return X_train, Y_train

  elif handleImbalance == 4 or handleImbalance == "skip":
    return X_train, Y_train

  else:
    print("Invalid Input")

  return X_train, Y_train

In [88]:
def remove_text_cols(X_train, X_test):
  X_train_num, _, _, _ = extract_numerical_text(X_train)
  X_test_num, _, _, _ = extract_numerical_text(X_test)

  return X_train_num, X_test_num

In [89]:
class Model:
  def __init__(self, name, model, std):
    self.name = name
    self.model = model
    self.std = std
    acc_score = None

In [90]:
def main():
  tSize = 0.2
  random_state = 69
  method_handling_missing_values = "drop"

  df = pd.read_csv("/content/drive/MyDrive/DataSets/Project 1 - SONAR Rock vs Mine/Copy of sonar data.csv", header=None)

  #
  section("DATA PREPROCESSING STATUS")

  label_number = -1

  df = handle_missing_values(df, method_handling_missing_values)

  df, X, Y, X_train, Y_train, X_test, Y_test, le = split(df, label_number, tSize, random_state)

  label_arr = df.iloc[:, -1].value_counts()
  label_count = dict(sorted(label_arr.items(), key=lambda x: x[1]))

  ##
  subsection("CLASS DISTRIBUTION")

  # print label count in label column
  for key, value in label_count.items():
    print(f"{key}: {value}")

  ##
  subsection("IMBALANCE HANDLING")

  print("Do You Need: ")
  print("     1) Undersample")
  print("     2) Oversample")
  print("     3) SMOTE")
  print("     4) Skip")
  method_handling_imbalance = int(input())

  X_train, Y_train = handle_imbalace(X_train, Y_train, method_handling_imbalance)

  models = []
  best_model = None
  max_acc_score = 0

  models.append(Model("LogisticRegression", LogisticRegression(max_iter=1000, random_state=random_state), True))
  models.append(Model("KNN", KNeighborsClassifier(n_neighbors=5), True))
  models.append(Model("DecisonTree", DecisionTreeClassifier(random_state=random_state), False))
  models.append(Model("SVM", SVC(kernel='linear'), True))
  models.append(Model("NaiveBayes", GaussianNB(), False))
  models.append(Model("Random Forest", RandomForestClassifier(n_estimators=100, random_state=random_state), False))

  ##
  subsection("MODEL TRAINING STATUS")

  for m in models:

    X_tr, X_te = remove_text_cols(X_train.copy(), X_test.copy())

    if m.std:
      X_tr, X_te= Standardize(X_tr, X_te)

    m.model.fit(X_tr, Y_train)
    success(m.name + " trained")

    Y_pred = m.model.predict(X_te)

    m.acc_score = accuracy_score(Y_test, Y_pred)

    if m.acc_score > max_acc_score:
      max_acc_score = m.acc_score
      best_model = m.name

  section("MODEL EVALUATION — ACCURACY")

  for m in models:
    print(f"{m.name} : {round((m.acc_score * 100), 2)}%")

  subsection("BEST MODEL")
  print(f"{best_model} with accuracy = {round((max_acc_score * 100), 2)}%")

In [91]:
if __name__ == "__main__":
  main()


DATA PREPROCESSING STATUS
✔ Missing values dropped
✔ Labels encoded
✔ Train-test split completed

--------------------------------------------------
CLASS DISTRIBUTION
--------------------------------------------------
R: 97
M: 111

--------------------------------------------------
IMBALANCE HANDLING
--------------------------------------------------
Do You Need: 
     1) Undersample
     2) Oversample
     3) SMOTE
     4) Skip
4

--------------------------------------------------
MODEL TRAINING STATUS
--------------------------------------------------
✔ Standardization applied
✔ LogisticRegression trained
✔ Standardization applied
✔ KNN trained
✔ DecisonTree trained
✔ Standardization applied
✔ SVM trained
✔ NaiveBayes trained
✔ Random Forest trained

MODEL EVALUATION — ACCURACY
LogisticRegression : 76.19%
KNN : 78.57%
DecisonTree : 61.9%
SVM : 76.19%
NaiveBayes : 61.9%
Random Forest : 78.57%

--------------------------------------------------
BEST MODEL
----------------------------